# Linear Regression

- by using Onehotencoding, StringIndexer, StandardScaler


In [ ]:
import os
import numpy as np
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

In [ ]:
spark = SparkSession.builder.appName('HousingModelDocker').getOrCreate()

In [ ]:
RECORD_TO_SHOW=2

## Step 1: Create Dataframe

In [121]:
data = spark.read.csv('work/housing_prices.csv', header=True, inferSchema=True)
data.printSchema()

root
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- housing_median_age: double (nullable = true)
 |-- total_rooms: double (nullable = true)
 |-- total_bedrooms: double (nullable = true)
 |-- population: double (nullable = true)
 |-- households: double (nullable = true)
 |-- median_income: double (nullable = true)
 |-- median_house_value: double (nullable = true)
 |-- ocean_proximity: string (nullable = true)



In [123]:
data.show(5)

+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|  -122.23|   37.88|              41.0|      880.0|         129.0|     322.0|     126.0|       8.3252|          452600.0|       NEAR BAY|
|  -122.22|   37.86|              21.0|     7099.0|        1106.0|    2401.0|    1138.0|       8.3014|          358500.0|       NEAR BAY|
|  -122.24|   37.85|              52.0|     1467.0|         190.0|     496.0|     177.0|       7.2574|          352100.0|       NEAR BAY|
|  -122.25|   37.85|              52.0|     1274.0|         235.0|     558.0|     219.0|       5.6431|          341300.0|       NEAR BAY|
|  -122.25|   37.85|              

##### Handling NULL values

In [ ]:
(
    data
        .select([
            F.count(
                F.when(
                    F.col(c).isNull(), 1
                )
            ).alias(c) for c in data.columns]
        ).show()
)

data = data.dropna(subset='total_bedrooms')

## Step 2: Create dataframes for training  and testing

- Using StringIndexer
- Using OneHotEncoder

In [ ]:
training_data, testing_data = data.randomSplit([0.8,0.2], seed=42)
print(f'Training data count: {training_data.count()}')
print(f'Testing data count: {testing_data.count()}')

In [ ]:
training_data.show(RECORD_TO_SHOW)

In [ ]:
testing_data.show(RECORD_TO_SHOW)

##### Features Improvement

- Using StringIndexer

In [ ]:
indexer = StringIndexer(inputCol='ocean_proximity', outputCol='ocean_proximity_index')
indexer_model = indexer.fit(training_data)
training_data = indexer_model.transform(training_data)
training_data.show(RECORD_TO_SHOW)

In [ ]:
training_data.select('ocean_proximity','ocean_proximity_index').distinct().show()

- Using OneHotEncoder

In [ ]:
encoder = OneHotEncoder(inputCol='ocean_proximity_index', outputCol='ocean_proximity_vec', dropLast=False)
encoder_model = encoder.fit(training_data)
training_data = encoder_model.transform(training_data)
training_data.show(RECORD_TO_SHOW)

In [ ]:
training_data.select('ocean_proximity','ocean_proximity_index','ocean_proximity_vec').distinct().show()

## Step 3: Create Feature Packing Dataframe

Machine learning models are like picky chefs:

- They want ALL ingredients in ONE bowl (unscaled_feature)
- Not separate bowls for age, rooms, population, ocean location

In [ ]:
feature_columns = [
    'housing_median_age',
    'total_rooms',
    'total_bedrooms',
    'population',
    'households',
    'median_income',
    'ocean_proximity_vec'
]

assembler = VectorAssembler(inputCols=feature_columns, outputCol='unscaled_features')
training_data = assembler.transform(training_data)
training_data.show(RECORD_TO_SHOW)

## Step 3: Makes numbers same scale (like standardizing units)

Imagine your toy cars have different wheel sizes!

Some cars are tiny (2cm wheels), some are huge (20cm wheels). When you race them, the big wheels always win because they roll faster!

🏎️ StandardScaler = "Wheel Size Equalizer"

In [ ]:
scaler = StandardScaler(inputCol='unscaled_features', outputCol='features',withMean=True, withStd=True)
scaler_model = scaler.fit(training_data)
transformed_training_data = scaler_model.transform(training_data)
transformed_training_data.show(RECORD_TO_SHOW)

In [ ]:
transformed_training_data.select('unscaled_features','features').show(RECORD_TO_SHOW)

## Step 4: Creating Model and Train it

In [ ]:
lr = LinearRegression(featuresCol='features',labelCol='median_house_value')
model = lr.fit(transformed_training_data)

## Step 5: Apply transformation to testing data

Note: Our training data apply StringIndexer, OneHotEncoding, VectorAssembler and StandardScaler so its very important to make the test data similar to train data in terms of transformation. 

In [ ]:
testing_data = indexer_model.transform(testing_data)
testing_data = encoder_model.transform(testing_data)
testing_data = assembler.transform(testing_data)
testing_data = scaler_model.transform(testing_data)
testing_data.show(RECORD_TO_SHOW)

## Step 6: Prediction

In [ ]:
test_prediction = model.transform(testing_data)
test_prediction.select('median_house_value','prediction').show(RECORD_TO_SHOW)

## Evaluation

In [ ]:
evaluator_mae = RegressionEvaluator(labelCol='median_house_value',predictionCol='prediction', metricName='mae')
mae = evaluator_mae.evaluate(test_prediction)
print(f'Mean Absolute Error (MAE): {mae}')